# AutoHDR-clone — Train 3 ứng viên trên Colab (30/07/2026)

**Cách chạy:** menu `Runtime` → `Change runtime type` → chọn **GPU (T4)** → `Runtime` → `Run all`.
Khi hỏi quyền Google Drive thì cho phép (đọc data + lưu checkpoint vào `MyDrive/autohdr_kit/`).

Notebook train tuần tự 3 ứng viên rồi render bộ chấm BENCH-10:
- **E1 CH_O**: HDRNet cũ + data mới (control)
- **E3 NM_A**: kiến trúc mới gain-map + 3D-LUT (tự viết)
- **E5 RES**: fine-tune Restormer pretrain (MIT) — model có sẵn tri thức, học thẳng cặp ảnh

Checkpoint ghi THẲNG vào Drive sau mỗi lần cải thiện → đứt phiên không mất gì, chạy lại tự nối tiếp.

In [3]:
# ── Cell 1: Môi trường + Drive + repo + data ──
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install opencv-python-headless scikit-image einops 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import os, sys, zipfile, shutil, glob
KIT = '/content/drive/MyDrive/autohdr_kit/colab'
OUT = '/content/drive/MyDrive/autohdr_kit/colab_out'
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(f'{KIT}/data_all.zip'), 'Thieu data_all.zip trong Drive autohdr_kit/colab'
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/phuocthanhiovn-coder/ai-nh-c-t.git /content/repo
%cd /content/repo
sys.path.insert(0, '/content/repo')
for z in ['data_all.zip', 'bench_in.zip']:
    if not os.path.exists(z.replace('.zip','')):
        zipfile.ZipFile(f'{KIT}/{z}').extractall('.')
os.makedirs('checkpoints/sweep', exist_ok=True)
shutil.copy(f'{KIT}/CH_M2.pt', 'checkpoints/sweep/CH_M2.pt')
print('data:', len(os.listdir('data_all/before')), 'cap |', 'bench:', len(os.listdir('bench_in')))

Tesla T4, 15360 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/repo
data: 138 cap | bench: 10


In [4]:
# ── Cell 2: E1 CH_O — HDRNet + data mới (control), ~30-50 phút T4 ──
from ai_engine.specialists.auto_enhance.gpu.train_sweep import train_one
import os
OUTP = f'{OUT}/CH_O.pt'
if os.path.exists(OUTP):
    print('CH_O da co tren Drive, bo qua (xoa file neu muon train lai)')
else:
    cfg = {
        'init_ckpt': 'checkpoints/sweep/CH_M2.pt', 'data_dir': 'data_all',
        'grid_bins': 10, 'grid_size': 24, 'proxy_res': 448, 'width': 32,
        'crop': 512, 'batch_size': 4, 'lr': 6e-5, 'epochs': 150,
        'loss': {'w_l1': 0.2, 'w_lab': 0.6, 'lab_weights': [0.55, 1.5, 1.5],
                 'w_perc': 0.08, 'w_hi': 0.2, 'hi_gamma': 2.0,
                 'w_dark': 0.6, 'dark_thresh': 0.28, 'w_color': 1.2, 'w_lc': 0.8},
        'amp': True, 'device': 'cuda', 'val_frac': 0.12,
        'num_workers': 2, 'cache_ram': True, 'cache_cap': 120,
        'out': OUTP, 'save_every': 20,
    }
    print('RESULT', train_one(cfg))

[*] run='CH_O'  device=cuda  amp=True  (requested amp=True)  cache_ram=True
[+] 127 train / 11 val pairs (val_frac=0.12)
[+] Val files (stable across runs): ['a_hr_fp104585.jpg', 'a_hr_fp104665.jpg', 'a_hr_fp104755.jpg', 'a_hr_fp104785.jpg', 'b_hr_dsc7289.jpg', 'b_hr_dsc7295.jpg', 'b_hr_dsc7384.jpg', 'b_hr_dsc7462.jpg', 'b_hr_dsc7580.jpg', 'b_hr_dsc7601.jpg', 'b_hr_dsc7603.jpg']
[+] RAM-cached 127 train + 11 val imgs (cap 120px)
[+] Warm-start tu 'checkpoints/sweep/CH_M2.pt' (missing=0, unexpected=0)


/content/repo/ai_engine/specialists/auto_enhance/gpu/train_sweep.py:415: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


[VGGPerceptual] Loading vgg16 (VGG16_Weights.DEFAULT); first run downloads ~528 MB to the torch hub cache...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 78.3MB/s]


  Epoch 001/150 | train_total=0.221143 val_total=0.248079 val_l1=0.084299 lr=1.200e-05 time=42.84s
  Epoch 002/150 | train_total=0.209144 val_total=0.240104 val_l1=0.080277 lr=2.400e-05 time=7.86s
  Epoch 003/150 | train_total=0.200975 val_total=0.234089 val_l1=0.076799 lr=3.600e-05 time=8.60s
  Epoch 004/150 | train_total=0.195462 val_total=0.213731 val_l1=0.076058 lr=4.800e-05 time=7.88s
  Epoch 005/150 | train_total=0.185749 val_total=0.212747 val_l1=0.075595 lr=6.000e-05 time=8.34s
  Epoch 006/150 | train_total=0.181720 val_total=0.217033 val_l1=0.068785 lr=5.999e-05 time=8.51s
  Epoch 007/150 | train_total=0.180735 val_total=0.216514 val_l1=0.067249 lr=5.997e-05 time=8.56s
  Epoch 008/150 | train_total=0.172649 val_total=0.216043 val_l1=0.066692 lr=5.994e-05 time=8.83s
  Epoch 009/150 | train_total=0.170665 val_total=0.206659 val_l1=0.065178 lr=5.989e-05 time=7.98s
  Epoch 010/150 | train_total=0.169211 val_total=0.205407 val_l1=0.068582 lr=5.982e-05 time=9.05s
  Epoch 011/150 | t

In [5]:
# ── Cell 3: E3 NM_A — kiến trúc mới gain-map + 3D-LUT, ~45-70 phút T4 ──
from ai_engine.specialists.auto_enhance.gpu.train_nm import train_nm
import os
OUTP = f'{OUT}/NM_A.pt'
if os.path.exists(OUTP):
    print('NM_A da co tren Drive, bo qua')
else:
    cfg = {
        'data_dir': 'data_all',
        'base': 32, 'n_basis': 6, 'lut_dim': 17, 'proxy_res': 448, 'gain_amp': 1.4,
        'crop': 512, 'batch_size': 4, 'lr': 2e-4, 'epochs': 200,
        'loss': {'w_l1': 0.2, 'w_lab': 0.6, 'lab_weights': [0.55, 1.5, 1.5],
                 'w_perc': 0.08, 'w_hi': 0.2, 'hi_gamma': 2.0,
                 'w_dark': 0.6, 'dark_thresh': 0.28, 'w_color': 1.2, 'w_lc': 0.8},
        'amp': True, 'device': 'cuda', 'val_frac': 0.12,
        'num_workers': 2, 'cache_ram': True, 'cache_cap': 120,
        'out': OUTP,
    }
    print('RESULT', train_nm(cfg))

[NM] 127 train / 11 val
[NM] params=2005051 kwargs={'base': 32, 'n_basis': 6, 'lut_dim': 17, 'proxy_res': 448, 'gain_amp': 1.4}


/content/repo/ai_engine/specialists/auto_enhance/gpu/train_nm.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


ValueError: too many values to unpack (expected 2)

In [ ]:
# ── Cell 4: E5 RES — fine-tune Restormer pretrain (MIT), ~60-90 phút T4 ──
# Model DA pretrain (26M params, real denoising — da biet 'anh sach' la gi)
# hoc thang cap anh cua minh. 50-138 cap la du vi chi day 'gu', khong day chup anh.
import os, sys, time, torch, torch.nn.functional as F
from torch.utils.data import DataLoader
OUTP = f'{OUT}/E5_RES.pt'
if not os.path.exists('/content/Restormer'):
    !git clone -q https://github.com/swz30/Restormer.git /content/Restormer
W = '/content/real_denoising.pth'
if not os.path.exists(W):
    !wget -q https://github.com/swz30/Restormer/releases/download/v1.0/real_denoising.pth -O {W} || echo TAI_WEIGHTS_LOI
sys.path.insert(0, '/content/Restormer/basicsr/models/archs')
from restormer_arch import Restormer
model = Restormer(LayerNorm_type='BiasFree')
if os.path.exists(W) and os.path.getsize(W) > 1e6:
    model.load_state_dict(torch.load(W, map_location='cpu')['params'])
    print('da nap pretrain real_denoising')
else:
    print('!! khong tai duoc weights — train tu dau (van chay, yeu hon)')
model = model.cuda()
if os.path.exists(OUTP):
    model.load_state_dict(torch.load(OUTP, map_location='cpu'))
    print('nap lai E5 tu Drive (chay tiep)')
from ai_engine.specialists.auto_enhance.gpu.train_sweep import CropPairDataset, split_filenames
from ai_engine.specialists.auto_enhance.gpu.losses import CombinedLoss
tr_f, va_f = split_filenames('data_all', 0.12)
tr = DataLoader(CropPairDataset('data_all', tr_f, 256, 64, is_train=True, cache_ram=True, cache_cap=120),
                batch_size=2, shuffle=True, num_workers=2)
va = DataLoader(CropPairDataset('data_all', va_f, 256, 64, is_train=False, cache_ram=True, cache_cap=40),
                batch_size=2, num_workers=2)
crit = CombinedLoss(w_l1=0.2, w_lab=0.6, lab_weights=(0.55, 1.5, 1.5), w_perc=0.08,
                    w_hi=0.2, hi_gamma=2.0, w_dark=0.6, dark_thresh=0.28,
                    w_color=1.2, w_lc=0.8).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler()
best = 1e9
for ep in range(1, 61):
    model.train(); t0 = time.time(); tl = n = 0
    for before, _, after in tr:
        before, after = before.cuda(), after.cuda()
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = crit(model(before).clamp(0, 1), after)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item() * before.size(0); n += before.size(0)
    model.eval(); vl = vn = 0
    with torch.no_grad(), torch.cuda.amp.autocast():
        for before, _, after in va:
            before, after = before.cuda(), after.cuda()
            vl += crit(model(before).clamp(0, 1), after).item() * before.size(0); vn += before.size(0)
    vt = vl / max(vn, 1)
    print(f'E5 ep {ep:03d}/060 train={tl/max(n,1):.4f} val={vt:.4f} {time.time()-t0:.0f}s', flush=True)
    if vt < best:
        best = vt
        torch.save(model.state_dict(), OUTP)   # ghi THANG vao Drive
print('E5 xong, best val', best)

In [ ]:
# ── Cell 5: Render BENCH-10 cho cả 3 ứng viên → Drive colab_out/render_r2/ ──
import os, sys, cv2, numpy as np, torch, torch.nn.functional as F
from ai_engine.specialists.auto_enhance.gpu.model_v2 import HDRNetV2
from ai_engine.specialists.auto_enhance.gpu.model_nm import NMNet
sys.path.insert(0, '/content/Restormer/basicsr/models/archs')
from restormer_arch import Restormer
KW = dict(grid_bins=10, grid_size=24, proxy_res=448, width=32)

def render_op(model, img):
    full = torch.from_numpy(img.transpose(2, 0, 1)).float()[None].cuda() / 255.0
    proxy = F.interpolate(full, size=(448, 448), mode='bilinear', align_corners=False)
    with torch.no_grad():
        out = model(proxy, full)
    return (out[0].clamp(0, 1).cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)

def render_res(model, img):
    # Restormer chay o ~1408px -> chung cat RATIO ap len full-res (hop dong operator)
    h, w = img.shape[:2]; s = 1408.0 / max(h, w)
    small = cv2.resize(img, (int(w*s)//8*8, int(h*s)//8*8), interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(small.transpose(2, 0, 1)).float()[None].cuda() / 255.0
    with torch.no_grad(), torch.cuda.amp.autocast():
        o = model(t).clamp(0, 1)
    o = o[0].float().cpu().numpy().transpose(1, 2, 0)
    ratio = (o + 1e-3) / (small.astype(np.float32) / 255.0 + 1e-3)
    ratio = cv2.resize(np.clip(ratio, 0.2, 5.0), (w, h), interpolation=cv2.INTER_LINEAR)
    return np.clip(img.astype(np.float32) / 255.0 * ratio, 0, 1) * 255

cands = []
if os.path.exists(f'{OUT}/CH_O.pt'):
    m = HDRNetV2(**KW); m.load_state_dict(torch.load(f'{OUT}/CH_O.pt', map_location='cpu'))
    cands.append(('CH_O', m.cuda().eval(), render_op))
if os.path.exists(f'{OUT}/NM_A.pt'):
    m = NMNet(base=32, n_basis=6, lut_dim=17, proxy_res=448, gain_amp=1.4)
    m.load_state_dict(torch.load(f'{OUT}/NM_A.pt', map_location='cpu'))
    cands.append(('NM_A', m.cuda().eval(), render_op))
if os.path.exists(f'{OUT}/E5_RES.pt'):
    m = Restormer(LayerNorm_type='BiasFree'); m.load_state_dict(torch.load(f'{OUT}/E5_RES.pt', map_location='cpu'))
    cands.append(('E5_RES', m.cuda().eval(), render_res))
print('render:', [c[0] for c in cands])
for f in sorted(os.listdir('bench_in')):
    img = cv2.imread(os.path.join('bench_in', f))
    for name, model, fn in cands:
        od = f'{OUT}/render_r2/{name}'; os.makedirs(od, exist_ok=True)
        dst = os.path.join(od, f)
        if os.path.exists(dst):
            continue
        out = fn(model, img).astype(np.uint8)
        cv2.imwrite(dst, out, [cv2.IMWRITE_JPEG_QUALITY, 95])
        print(name, f, flush=True)
print('XONG HET — bao Claude keo ve dong goi cham mu')